# Visualizations

This notebook prototypes all charts for the UCSB Rental Price Streamlit app. Each chart is built in **Plotly** so the code can be ported directly into `streamlit_app.py` with minimal changes.

**Audience:** Students looking for housing near UCSB  
**Charts in this notebook:**
1. Median rent by bedroom count
2. Price distribution by bedroom (box plot)
3. Price vs. distance to UCSB (scatter, filterable)
4. Listing availability heatmap (distance × bedrooms)
5. Rent predictor output (range bar)

## Setup

Imports, data loading, and model loading. The trained model is used only for Chart 5 (the predictor output).

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.ensemble import GradientBoostingRegressor

# ── Load data ──────────────────────────────────────────────────────────────
df = pd.read_csv(
    "../data/cleaned/master_ucsb_cleaned_listings.csv",
    encoding="latin1"
)
df = df[df['price_from'] > 0].copy()

# Human-readable bedroom labels
BED_LABELS = {0: 'Studio', 1: '1 bed', 2: '2 bed', 3: '3 bed',
              4: '4 bed', 5: '5 bed', 6: '6 bed', 7: '7 bed'}
df['bed_label'] = df['bedrooms'].map(BED_LABELS)

print(f"Dataset: {len(df)} listings")
df.head()

In [ ]:
# ── Load model (trained in 03_modeling_gradient_boosting.ipynb) ────────────
# Option A — re-train inline (fine for prototyping)
model = GradientBoostingRegressor(
    n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42
)
model.fit(
    df[['bedrooms', 'bathrooms', 'distance_to_ucsb_miles']],
    df['price_from']
)

# Option B — load a saved pickle once you export it from notebook 03
# import pickle
# with open("../models/rental_model.pkl", "rb") as f:
#     model = pickle.load(f)

print("Model ready")

---
## Chart 1 — Median Rent by Bedroom Count

**Purpose:** The first thing a student wants to know — "what does a 2-bedroom typically cost?"

**Streamlit placement:** Homepage / top of market overview section.  
**Streamlit note:** Copy this cell's code directly. No changes needed.

In [ ]:
bed_stats = (
    df.groupby(['bedrooms', 'bed_label'])['price_from']
    .agg(median_rent='median', count='count')
    .reset_index()
    .sort_values('bedrooms')
)

fig1 = px.bar(
    bed_stats,
    x='bed_label',
    y='median_rent',
    text=bed_stats['median_rent'].apply(lambda v: f"${v:,.0f}"),
    custom_data=['count'],
    labels={'bed_label': 'Bedroom type', 'median_rent': 'Median monthly rent ($)'},
    title='Median monthly rent by bedroom count',
    color_discrete_sequence=['#4C78A8']
)

fig1.update_traces(
    textposition='outside',
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Median rent: $%{y:,.0f}<br>"
        "Listings in dataset: %{customdata[0]}"
        "<extra></extra>"
    )
)
fig1.update_layout(
    yaxis_tickprefix='$',
    yaxis_tickformat=',',
    plot_bgcolor='white',
    yaxis=dict(gridcolor='#eeeeee')
)

fig1.show()

**Reading this chart:** 6-bedroom listings show an unexpectedly low median (~$1,660). This is a known data quality issue — some multi-bedroom listings appear to be priced per-person rather than as a total unit. The app should note this caveat for 6+ bedroom results.

---
## Chart 2 — Price Distribution by Bedroom (Box Plot)

**Purpose:** Shows the full spread of prices within each bedroom tier, not just the median. Students can see best-case, typical, and worst-case rent for their target unit size.

**Streamlit placement:** Market overview section, below Chart 1.  
**Streamlit note:** Consider adding a `st.multiselect` to filter which bedroom types are shown.

In [ ]:
# Filter to 0–5 bedrooms — 6 and 7 have too few listings to be meaningful
df_box = df[df['bedrooms'] <= 5].copy()
# Preserve bedroom order on x-axis
bed_order = [BED_LABELS[i] for i in range(6)]

fig2 = px.box(
    df_box,
    x='bed_label',
    y='price_from',
    category_orders={'bed_label': bed_order},
    points='all',
    labels={'bed_label': 'Bedroom type', 'price_from': 'Monthly rent ($)'},
    title='Rent price distribution by bedroom count',
    color='bed_label',
    color_discrete_sequence=px.colors.qualitative.Pastel
)

fig2.update_traces(
    hovertemplate="$%{y:,.0f}<extra></extra>"
)
fig2.update_layout(
    showlegend=False,
    yaxis_tickprefix='$',
    yaxis_tickformat=',',
    plot_bgcolor='white',
    yaxis=dict(gridcolor='#eeeeee')
)

fig2.show()

---
## Chart 3 — Price vs. Distance to UCSB

**Purpose:** Shows whether proximity to campus commands a price premium, and lets students explore the trade-off between price and commute distance.

**Streamlit placement:** Market overview section.  
**Streamlit note:** The `color='bed_label'` already creates a legend that doubles as a filter in Plotly. In Streamlit, pair with a `st.multiselect` for bedroom type to let students isolate their tier.

In [ ]:
df_scatter = df[df['bedrooms'] <= 5].copy()
bed_order = [BED_LABELS[i] for i in range(6)]

fig3 = px.scatter(
    df_scatter,
    x='distance_to_ucsb_miles',
    y='price_from',
    color='bed_label',
    category_orders={'bed_label': bed_order},
    hover_data={'address': True, 'bathrooms': True,
                'distance_to_ucsb_miles': ':.2f',
                'price_from': ':,.0f',
                'bed_label': False},
    labels={
        'distance_to_ucsb_miles': 'Distance to UCSB (miles)',
        'price_from': 'Monthly rent ($)',
        'bed_label': 'Bedroom type'
    },
    title='Rent vs. distance to UCSB',
    opacity=0.75
)

fig3.update_layout(
    yaxis_tickprefix='$',
    yaxis_tickformat=',',
    plot_bgcolor='white',
    xaxis=dict(gridcolor='#eeeeee'),
    yaxis=dict(gridcolor='#eeeeee')
)

fig3.show()

---
## Chart 4 — Listing Availability Heatmap

**Purpose:** Answers "where can I actually find a 2-bedroom within 1 mile?" — shows inventory density by distance band and bedroom count.

**Streamlit placement:** Market overview section, optionally collapsed under an expander.  
**Streamlit note:** Copy directly. The `colorscale` can be swapped to match your app's theme.

In [ ]:
df['dist_band'] = pd.cut(
    df['distance_to_ucsb_miles'],
    bins=[0, 0.5, 1.0, 1.5, 2.0, 10],
    labels=['< 0.5 mi', '0.5–1 mi', '1–1.5 mi', '1.5–2 mi', '2+ mi']
)

heat = (
    df[df['bedrooms'] <= 5]
    .groupby(['dist_band', 'bedrooms'], observed=True)
    .size()
    .reset_index(name='count')
)

heat_pivot = (
    heat.pivot(index='dist_band', columns='bedrooms', values='count')
    .fillna(0)
    .astype(int)
)

# Rename columns to readable labels
heat_pivot.columns = [BED_LABELS[c] for c in heat_pivot.columns]

fig4 = go.Figure(data=go.Heatmap(
    z=heat_pivot.values,
    x=heat_pivot.columns.tolist(),
    y=heat_pivot.index.tolist(),
    colorscale='Blues',
    text=heat_pivot.values,
    texttemplate='%{text}',
    hovertemplate=(
        "<b>%{y} · %{x}</b><br>"
        "Listings: %{z}"
        "<extra></extra>"
    )
))

fig4.update_layout(
    title='Number of listings by distance band and bedroom count',
    xaxis_title='Bedroom type',
    yaxis_title='Distance to UCSB'
)

fig4.show()

---
## Chart 5 — Rent Predictor Output

**Purpose:** The core UI element of the app — shows students a price estimate with a confidence range for their specific search criteria.

**Streamlit placement:** Predictor section. The inputs (`bedrooms`, `bathrooms`, `distance`) will come from `st.selectbox` / `st.slider` widgets. The chart re-renders live when inputs change.  
**Streamlit note:** Wrap the chart code in a function `render_prediction_chart(bedrooms, bathrooms, distance)` and call it inside the Streamlit app.

In [ ]:
def predict_rent(bedrooms, bathrooms, distance_to_ucsb_miles):
    """
    Returns a point estimate and confidence range.
    Wider margin for 3+ bedrooms due to higher data variance.
    """
    features = pd.DataFrame([{
        'bedrooms': bedrooms,
        'bathrooms': bathrooms,
        'distance_to_ucsb_miles': distance_to_ucsb_miles
    }])
    estimate = round(model.predict(features)[0])
    margin = 2500 if bedrooms >= 3 else 850
    return {
        'estimate': estimate,
        'low':  max(0, estimate - margin),
        'high': estimate + margin,
        'margin': margin
    }


def render_prediction_chart(bedrooms, bathrooms, distance):
    """
    Renders a horizontal range bar showing the predicted rent and confidence interval.
    Copy this function directly into streamlit_app.py.
    Call: fig = render_prediction_chart(bedrooms, bathrooms, distance)
          st.plotly_chart(fig, use_container_width=True)
    """
    pred = predict_rent(bedrooms, bathrooms, distance)
    bed_label = BED_LABELS.get(bedrooms, f'{bedrooms} bed')

    fig = go.Figure()

    # Confidence range bar
    fig.add_trace(go.Bar(
        x=[pred['high'] - pred['low']],
        y=['Estimated rent'],
        base=[pred['low']],
        orientation='h',
        marker_color='#AEC6E8',
        hovertemplate=(
            f"Range: ${pred['low']:,} – ${pred['high']:,}"
            "<extra>Confidence range</extra>"
        ),
        name='Confidence range',
        width=0.4
    ))

    # Point estimate marker
    fig.add_trace(go.Scatter(
        x=[pred['estimate']],
        y=['Estimated rent'],
        mode='markers+text',
        marker=dict(color='#1F5FA6', size=18, symbol='diamond'),
        text=[f"  ${pred['estimate']:,}/mo"],
        textposition='middle right',
        textfont=dict(size=16, color='#1F5FA6'),
        hovertemplate=f"Estimate: ${pred['estimate']:,}/mo<extra></extra>",
        name='Estimate'
    ))

    note = (
        f"Typical error ±$2,500 for 3+ bedroom listings — price data varies widely in this segment."
        if bedrooms >= 3
        else f"Typical prediction error ±$850."
    )

    fig.update_layout(
        title=dict(
            text=f"Predicted rent · {bed_label} · {bathrooms} bath · {distance} mi from UCSB",
            font=dict(size=15)
        ),
        xaxis=dict(
            title='Monthly rent ($)',
            tickprefix='$',
            tickformat=',',
            gridcolor='#eeeeee'
        ),
        yaxis=dict(showticklabels=False),
        plot_bgcolor='white',
        showlegend=True,
        legend=dict(orientation='h', y=-0.25),
        height=220,
        annotations=[dict(
            text=note,
            xref='paper', yref='paper',
            x=0, y=-0.45,
            showarrow=False,
            font=dict(size=11, color='gray'),
            align='left'
        )],
        margin=dict(b=80)
    )
    return fig


# ── Preview with a few example inputs ──────────────────────────────────────
render_prediction_chart(bedrooms=1, bathrooms=1.0, distance=0.8).show()
render_prediction_chart(bedrooms=2, bathrooms=2.0, distance=0.5).show()
render_prediction_chart(bedrooms=3, bathrooms=2.0, distance=1.0).show()

---
## Streamlit Porting Guide

Below is a skeleton of how these charts map to a Streamlit app. This is for reference — the actual app lives in `streamlit_app.py`.

```python
import streamlit as st
import pickle
# ... (paste imports from Setup cell above)

# Load data and model
df = pd.read_csv("data/cleaned/master_ucsb_cleaned_listings.csv", encoding="latin1")
df = df[df['price_from'] > 0].copy()
with open("models/rental_model.pkl", "rb") as f:
    model = pickle.load(f)

st.title("UCSB Rental Price Explorer")

# ── Section 1: Market overview ───────────────────────────────────────────
st.header("Market overview")
st.plotly_chart(fig1, use_container_width=True)   # Chart 1 — median by bedrooms
st.plotly_chart(fig2, use_container_width=True)   # Chart 2 — box plot

bedroom_filter = st.multiselect(
    "Filter by bedroom type",
    options=['Studio','1 bed','2 bed','3 bed','4 bed','5 bed'],
    default=['Studio','1 bed','2 bed','3 bed']
)
df_filtered = df[df['bed_label'].isin(bedroom_filter)]
# re-render fig3 with df_filtered instead of df_scatter
st.plotly_chart(fig3, use_container_width=True)   # Chart 3 — scatter

with st.expander("Listing availability by area"):
    st.plotly_chart(fig4, use_container_width=True)  # Chart 4 — heatmap

# ── Section 2: Rent predictor ────────────────────────────────────────────
st.header("Estimate rent for a listing")

col1, col2, col3 = st.columns(3)
with col1:
    bedrooms = st.selectbox("Bedrooms", [0,1,2,3,4,5],
                            format_func=lambda x: BED_LABELS[x])
with col2:
    bathrooms = st.selectbox("Bathrooms", [1.0, 1.5, 2.0, 2.5, 3.0])
with col3:
    distance = st.slider("Distance to UCSB (miles)", 0.3, 5.0, 0.8, step=0.1)

fig5 = render_prediction_chart(bedrooms, bathrooms, distance)
st.plotly_chart(fig5, use_container_width=True)   # Chart 5 — predictor
```